# FLUX ARC Prize 2026 Submission

**Physics-Based Reasoning for ARC-AGI-2**

This notebook implements the FLUX approach to ARC:
- Zero forgetting between tasks
- O(log n) gravitational relevance (vs O(n²) attention)
- Thermodynamic settling for rule induction
- Delta wave encoding for transformations

---

## Cell 1: Setup & Clone Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX')

if ROOT.exists():
    print('Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

## Cell 2: Configure Paths

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX')
PHASES_DIR = ROOT / 'phases'
ARC_DIR = PHASES_DIR / 'phase9_arc'

# Add paths
for p in [ROOT, ARC_DIR, PHASES_DIR / 'phase8_8']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('✓ Paths configured')

## Cell 3: Load ARC Dataset

In [ ]:
import json
from pathlib import Path

# Kaggle provides ARC data in /kaggle/input/
ARC_DATA = Path('/kaggle/input/arc-prize-2025')

if not ARC_DATA.exists():
    # Fallback for local testing
    ARC_DATA = Path('/kaggle/input/arc-agi')

if not ARC_DATA.exists():
    print('⚠ ARC dataset not found, will use synthetic tasks')
    USE_SYNTHETIC = True
else:
    print(f'✓ ARC data found at {ARC_DATA}')
    USE_SYNTHETIC = False
    
    # List available files
    for f in sorted(ARC_DATA.glob('*'))[:10]:
        print(f'  {f.name}')

## Cell 4: Initialize FLUX Solver

In [ ]:
import torch
from arc_solver import ARCSolver
from arc_loader import ARCTask, ARCExample, load_arc_agi, generate_synthetic_tasks

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({vram:.1f} GB)')

# Initialize solver
solver = ARCSolver(use_waves=False, device=device, verbose=True)
print('✓ FLUX ARC Solver initialized')

## Cell 5: Load Evaluation Tasks

In [ ]:
from arc_loader import ARCDataset, ARCTask, ARCExample
import json

def load_kaggle_arc_tasks(data_dir: Path, split: str = 'challenges') -> ARCDataset:
    """Load ARC tasks from Kaggle competition format."""
    tasks = []
    
    # Try different possible file locations
    possible_paths = [
        data_dir / f'{split}.json',
        data_dir / 'arc-agi-training-challenges.json',
        data_dir / 'arc-agi-evaluation-challenges.json',
    ]
    
    challenges_file = None
    for p in possible_paths:
        if p.exists():
            challenges_file = p
            break
    
    if challenges_file:
        with open(challenges_file) as f:
            data = json.load(f)
        
        for task_id, task_data in data.items():
            train_examples = [
                ARCExample(input=ex['input'], output=ex['output'])
                for ex in task_data.get('train', [])
            ]
            test_examples = [
                ARCExample(input=ex['input'], output=ex.get('output', []))
                for ex in task_data.get('test', [])
            ]
            
            tasks.append(ARCTask(
                task_id=task_id,
                train=train_examples,
                test=test_examples,
                source='kaggle',
            ))
        
        print(f'✓ Loaded {len(tasks)} tasks from {challenges_file.name}')
    else:
        print(f'⚠ No challenges file found')
    
    return ARCDataset(name=split, tasks=tasks)

# Load tasks
if USE_SYNTHETIC:
    eval_tasks = generate_synthetic_tasks(20)
    print(f'Using {len(eval_tasks)} synthetic tasks')
else:
    eval_tasks = load_kaggle_arc_tasks(ARC_DATA, split='challenges')
    print(f'Loaded {len(eval_tasks)} evaluation tasks')

## Cell 6: Quick Validation

In [ ]:
# Test on first few tasks
print('Quick validation on first 5 tasks:')
print('=' * 50)

for task in list(eval_tasks)[:5]:
    print(f'\nTask: {task.task_id}')
    print(f'  Train examples: {task.num_train}')
    print(f'  Test examples: {task.num_test}')
    
    solution = solver.solve(task)
    
    if solution.hypotheses:
        print(f'  Best hypothesis: {solution.hypotheses[0].explanation}')
    print(f'  Output shape: {len(solution.attempt_1)}x{len(solution.attempt_1[0]) if solution.attempt_1 else 0}')
    print(f'  Time: {solution.solve_time_ms:.1f}ms')

## Cell 7: Solve All Tasks

In [ ]:
%%time
print('Solving all evaluation tasks...')
print('=' * 50)

solutions = solver.solve_all(eval_tasks)

# Summary
total_time = sum(s.solve_time_ms for s in solutions)
avg_time = total_time / len(solutions) if solutions else 0

print(f'\n✓ Solved {len(solutions)} tasks')
print(f'  Total time: {total_time/1000:.1f}s')
print(f'  Avg time/task: {avg_time:.1f}ms')

## Cell 8: Generate Submission

In [ ]:
import json
from pathlib import Path

# Generate submission file
submission = {}

for solution in solutions:
    submission[solution.task_id] = [
        {"attempt_1": solution.attempt_1}
    ]
    if solution.attempt_2:
        submission[solution.task_id][0]["attempt_2"] = solution.attempt_2

# Save submission
output_path = Path('/kaggle/working/submission.json')
with open(output_path, 'w') as f:
    json.dump(submission, f)

print(f'✓ Submission saved to {output_path}')
print(f'  Tasks: {len(submission)}')
print(f'  File size: {output_path.stat().st_size / 1024:.1f} KB')

## Cell 9: Evaluate (if ground truth available)

In [ ]:
# If running on training data with ground truth
if USE_SYNTHETIC or any(task.test and task.test[0].output for task in eval_tasks):
    results = solver.evaluate(solutions, eval_tasks)
    
    print('Evaluation Results:')
    print('=' * 50)
    print(f'  Total tasks: {results["total_tasks"]}')
    print(f'  Correct (attempt 1): {results["correct_attempt_1"]} ({results["accuracy_1"]*100:.1f}%)')
    print(f'  Correct (attempt 2): {results["correct_attempt_2"]} ({results["accuracy_2"]*100:.1f}%)')
    print(f'  Correct (either): {results["correct_either"]} ({results["accuracy_either"]*100:.1f}%)')
else:
    print('No ground truth available for evaluation')

## Cell 10: Analysis - Which Patterns Worked?

In [ ]:
from collections import Counter

# Analyze which patterns were identified
pattern_usage = Counter()
no_hypothesis = 0

for solution in solutions:
    if solution.hypotheses:
        for h in solution.hypotheses[:1]:  # Best hypothesis
            for pid in h.pattern_ids:
                pattern_usage[pid] += 1
    else:
        no_hypothesis += 1

print('Pattern Usage Analysis:')
print('=' * 50)
print(f'  Tasks with no hypothesis: {no_hypothesis}')
print(f'\nMost used patterns:')
for pid, count in pattern_usage.most_common(10):
    from pattern_library import PATTERNS
    if pid in PATTERNS:
        print(f'  {pid}: {PATTERNS[pid].name} ({count} tasks)')

## Cell 11: Visualize Sample Solutions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Color map for ARC (0-9)
ARC_COLORS = [
    '#000000',  # 0: black
    '#0074D9',  # 1: blue
    '#FF4136',  # 2: red
    '#2ECC40',  # 3: green
    '#FFDC00',  # 4: yellow
    '#AAAAAA',  # 5: gray
    '#F012BE',  # 6: magenta
    '#FF851B',  # 7: orange
    '#7FDBFF',  # 8: cyan
    '#870C25',  # 9: maroon
]

from matplotlib.colors import ListedColormap
arc_cmap = ListedColormap(ARC_COLORS)

def show_grid(ax, grid, title=''):
    """Display ARC grid."""
    arr = np.array(grid)
    ax.imshow(arr, cmap=arc_cmap, vmin=0, vmax=9)
    ax.set_title(title)
    ax.axis('off')
    
    # Add grid lines
    for i in range(arr.shape[0] + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5)
    for j in range(arr.shape[1] + 1):
        ax.axvline(j - 0.5, color='gray', linewidth=0.5)

# Show first 3 tasks with solutions
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, (task, solution) in enumerate(zip(list(eval_tasks)[:3], solutions[:3])):
    # Training example
    if task.train:
        show_grid(axes[row, 0], task.train[0].input, 'Train Input')
        show_grid(axes[row, 1], task.train[0].output, 'Train Output')
    
    # Test input
    if task.test:
        show_grid(axes[row, 2], task.test[0].input, 'Test Input')
    
    # Our prediction
    show_grid(axes[row, 3], solution.attempt_1, 'FLUX Prediction')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_solutions.png', dpi=150)
plt.show()

print('✓ Saved visualization to sample_solutions.png')

## Cell 12: Final Summary

In [ ]:
print('=' * 60)
print('FLUX ARC PRIZE 2026 SUBMISSION')
print('=' * 60)
print()
print('Model: FLUX (Field-based Latent Understanding eXperience)')
print('Approach: Physics-inspired reasoning')
print('  - Thermodynamic settling for rule induction')
print('  - Delta wave encoding for transformations')
print('  - O(log n) gravitational relevance')
print('  - Zero forgetting between tasks')
print()
print(f'Tasks solved: {len(solutions)}')
print(f'Average time: {avg_time:.1f}ms per task')
print()
print('Submission file: /kaggle/working/submission.json')
print('=' * 60)